<a href="https://colab.research.google.com/github/khagenA/eng_labs_marry_me/blob/lite/marry_me.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os

BASE_DIR = "/content/drive/MyDrive"
PROJECT = "QWASAR/eng_labs_marry_me"

os.chdir(os.path.join(BASE_DIR, PROJECT))
print("Current directory:", os.getcwd())


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Current directory: /content/drive/MyDrive/QWASAR/eng_labs_marry_me


In [2]:
import asyncio, json, time
from dataclasses import dataclass

WORK_SEC = 3.0
SIM_SEC  = 60.0

DEADLINE = {"high": 5.0, "medium": 10.0, "low": 15.0}

TYPE_TO_TEAM = {
    "brawl": "Security", "not_on_list": "Security",
    "bad_food": "Catering", "feeling_ill": "Catering",
    "dirty_table": "Waiters", "broken_item": "Waiters",
}

@dataclass
class Staff:
    status: str   # "Idle" | "Working"
    team: str

@dataclass(frozen=True)
class Event:
    id: int
    event_type: str
    priority: str
    description: str
    timestamp: float

def tnow(start): return time.monotonic() - start
def log(start, msg): print(f"[{tnow(start):6.2f}s] {msg}", flush=True)

async def worker_loop(staff: Staff, queue: asyncio.Queue, start, metrics):
    while True:
        ev = await queue.get()
        now = tnow(start)

        # expire if we only start looking at it after deadline
        if now > ev.timestamp + DEADLINE[ev.priority]:
            metrics["expired"] += 1
            metrics["stress"] += 1
            log(start, f"EXPIRED  #{ev.id} team={staff.team}")
            queue.task_done()
            continue

        staff.status = "Working"
        log(start, f"START    #{ev.id} team={staff.team} by={staff.team}-worker")
        await asyncio.sleep(WORK_SEC)
        staff.status = "Idle"
        metrics["handled"] += 1
        log(start, f"HANDLED  #{ev.id} team={staff.team}")
        queue.task_done()

async def run_sim(path="events.json", workers_per_team=2):
    events = [Event(**e) for e in json.load(open(path))]
    events.sort(key=lambda e: e.timestamp)

    start = time.monotonic()
    metrics = {"received": 0, "handled": 0, "expired": 0, "stress": 0}

    # queues per team
    queues = {team: asyncio.Queue() for team in {"Security", "Catering", "Waiters"}}

    # worker pool: N workers per team (concurrent handling)
    for team in queues:
        for _ in range(workers_per_team):
            asyncio.create_task(worker_loop(Staff("Idle", team), queues[team], start, metrics))

    log(start, "Simulation started")

    # ingest events at their timestamps
    for ev in events:
        await asyncio.sleep(max(0.0, ev.timestamp - tnow(start)))
        metrics["received"] += 1

        team = TYPE_TO_TEAM.get(ev.event_type)
        if (team is None) or (ev.priority not in DEADLINE):
            log(start, f"INVALID  #{ev.id} type={ev.event_type} prio={ev.priority} -> SKIP")
            continue

        log(start, f"RECEIVED #{ev.id} -> {team}")
        await queues[team].put(ev)

    # let sim run full duration
    await asyncio.sleep(max(0.0, SIM_SEC - tnow(start)))

    print("\nSUMMARY", flush=True)
    print("received:", metrics["received"], flush=True)
    print("handled :", metrics["handled"], flush=True)
    print("expired :", metrics["expired"], flush=True)
    print("stress  :", metrics["stress"], flush=True)


In [3]:
await run_sim("events_easy.json", workers_per_team=2)

[  0.00s] Simulation started
[  0.10s] RECEIVED #1 -> Waiters
[  0.11s] START    #1 team=Waiters by=Waiters-worker
[  1.60s] RECEIVED #2 -> Security
[  1.60s] START    #2 team=Security by=Security-worker
[  3.11s] HANDLED  #1 team=Waiters
[  4.61s] HANDLED  #2 team=Security
[  5.20s] RECEIVED #3 -> Security
[  5.20s] START    #3 team=Security by=Security-worker
[  7.40s] RECEIVED #4 -> Catering
[  7.40s] START    #4 team=Catering by=Catering-worker
[  8.20s] HANDLED  #3 team=Security
[ 10.41s] HANDLED  #4 team=Catering
[ 14.00s] RECEIVED #5 -> Catering
[ 14.01s] START    #5 team=Catering by=Catering-worker
[ 17.01s] HANDLED  #5 team=Catering
[ 23.81s] RECEIVED #6 -> Waiters
[ 23.81s] START    #6 team=Waiters by=Waiters-worker
[ 24.50s] RECEIVED #7 -> Catering
[ 24.50s] START    #7 team=Catering by=Catering-worker
[ 24.70s] RECEIVED #8 -> Catering
[ 24.70s] START    #8 team=Catering by=Catering-worker
[ 26.81s] HANDLED  #6 team=Waiters
[ 27.20s] RECEIVED #9 -> Catering
[ 27.50s] HANDLED

In [4]:
await run_sim("events_medium.json", workers_per_team=2)


[  0.00s] Simulation started
[  0.40s] RECEIVED #1 -> Security
[  0.40s] START    #1 team=Security by=Security-worker
[  1.40s] RECEIVED #2 -> Waiters
[  1.40s] START    #2 team=Waiters by=Waiters-worker
[  1.50s] RECEIVED #3 -> Security
[  1.50s] START    #3 team=Security by=Security-worker
[  1.60s] RECEIVED #4 -> Waiters
[  1.60s] START    #4 team=Waiters by=Waiters-worker
[  3.41s] HANDLED  #1 team=Security
[  4.41s] HANDLED  #2 team=Waiters
[  4.50s] HANDLED  #3 team=Security
[  4.60s] HANDLED  #4 team=Waiters
[  4.80s] RECEIVED #5 -> Waiters
[  4.80s] START    #5 team=Waiters by=Waiters-worker
[  5.10s] RECEIVED #6 -> Security
[  5.10s] START    #6 team=Security by=Security-worker
[  5.30s] RECEIVED #7 -> Catering
[  5.30s] START    #7 team=Catering by=Catering-worker
[  7.81s] HANDLED  #5 team=Waiters
[  8.11s] HANDLED  #6 team=Security
[  8.30s] HANDLED  #7 team=Catering
[  8.60s] RECEIVED #8 -> Security
[  8.60s] START    #8 team=Security by=Security-worker
[ 10.90s] RECEIVED 

In [5]:
await run_sim("events_hard.json", workers_per_team=2)


[  0.00s] Simulation started
[  0.10s] RECEIVED #1 -> Security
[  0.10s] START    #1 team=Security by=Security-worker
[  1.20s] RECEIVED #2 -> Security
[  1.20s] START    #2 team=Security by=Security-worker
[  1.40s] RECEIVED #3 -> Security
[  1.60s] RECEIVED #4 -> Catering
[  1.60s] START    #4 team=Catering by=Catering-worker
[  1.60s] RECEIVED #5 -> Catering
[  1.60s] START    #5 team=Catering by=Catering-worker
[  1.70s] RECEIVED #6 -> Waiters
[  1.70s] START    #6 team=Waiters by=Waiters-worker
[  1.90s] RECEIVED #7 -> Waiters
[  1.90s] START    #7 team=Waiters by=Waiters-worker
[  3.10s] HANDLED  #1 team=Security
[  3.11s] START    #3 team=Security by=Security-worker
[  4.21s] HANDLED  #2 team=Security
[  4.60s] HANDLED  #4 team=Catering
[  4.60s] HANDLED  #5 team=Catering
[  4.71s] HANDLED  #6 team=Waiters
[  4.90s] HANDLED  #7 team=Waiters
[  5.20s] RECEIVED #8 -> Security
[  5.20s] START    #8 team=Security by=Security-worker
[  6.11s] HANDLED  #3 team=Security
[  6.60s] RECEI